# Asistente de campañas sociales — Ben & Frank (Q3 2026)
**Tendencias de IA Generativa · ICESI**

LLM **local** (Ollama, Qwen 2.5 7B) + **RAG** (Chroma) + **LangChain / LangGraph** + **LangSmith** (trazas y evaluación).
El asistente existe en dos arquitecturas que se comparan con el mismo dataset:

- **Agente único (v5):** un agente ReAct con 5 herramientas y guardarraíles deterministas.
- **Multiagente:** supervisor (router) → especialistas (Datos, Anomalías, Conocimiento) → crítico (evaluador–optimizador).

Datos **sintéticos** que simulan un pull diario del data lake de social (jul–sep 2026; US, CA, PR; Facebook, TikTok, Snapchat, Pinterest, Reddit).

**Contenido**
1. Configuración
2. Modelos locales
3. Datos e índices
4. Agente único (v5)
5. Memoria conversacional
6. Arquitectura multiagente
7. Cálculo de costo
8. Evaluación comparativa en LangSmith
9. Extra: informe ejecutivo con salida estructurada
10. Anexo: depuración de argumentos

> **Requisitos:** Ollama con `qwen2.5:7b` (todos los agentes), `llama3.1:8b` (juez de evaluación) y `bge-m3` (embeddings);
> el CSV en `data/raw/`; el archivo `.env` en la raíz con `LANGSMITH_API_KEY`.

## 1. Configuración

In [ ]:
# Solo la primera vez (o si cambió requirements.txt)
# %pip install -qU -r ../requirements.txt

In [ ]:
import os, sys, getpass
from pathlib import Path

# El notebook vive en notebooks/: subimos a la raíz para importar src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# src.config carga el .env de la raíz (LANGSMITH_*, y llaves opcionales del juez)
from src.config import (LLM_PRINCIPAL, LLM_JUEZ_LOCAL, EMBEDDINGS, LANGSMITH_PROJECT, DATA_RAW,
                        MODELO_REFERENCIA_SOMBRA, TOPE_USD_POR_TAREA, MAX_RONDAS_CRITICO)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LANGSMITH_API_KEY: ")

import pandas as pd
import plotly.io as pio
pd.set_option("display.max_colwidth", 120)

print("Raíz:", ROOT)
print("LLM de los agentes:", LLM_PRINCIPAL, "| juez local:", LLM_JUEZ_LOCAL, "| embeddings:", EMBEDDINGS)

## 2. Modelos locales
Todo corre en la GPU local (RTX 3060, 12 GB): ningún dato sale a una API de LLM externa; solo las trazas van a LangSmith.

In [ ]:
import subprocess
from langchain.chat_models import init_chat_model

print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

model = init_chat_model(LLM_PRINCIPAL, temperature=0)
print(model.invoke("Explica en una frase qué es el CPM en publicidad digital.").content)

## 3. Datos e índices
Usa el CSV de `data/raw/`. La base DuckDB y el índice Chroma se construyen **solo si no existen**.

In [ ]:
DUCKDB = ROOT / "data/processed/campanas.duckdb"
CHROMA = ROOT / "data/processed/chroma"
assert Path(DATA_RAW).exists(), f"No encuentro {DATA_RAW}. Corre: python -m src.generar_datos"

if not DUCKDB.exists():
    from src.ingest import ingest
    ingest()
if not (CHROMA.exists() and any(CHROMA.iterdir())):
    from src.rag import construir_indice
    construir_indice()

# Capa determinista: totales de la campaña desde DuckDB
from src import tools as T
T.consultar_metricas()[["media_cost", "impressions", "clicks", "subscriptions", "ctr", "costo_por_cita", "vtr"]]

In [ ]:
# RAG
from src.rag import retriever
for d in retriever(k=2).invoke("¿cuándo una tasa LPV es preocupante?"):
    print(d.metadata.get("fuente"), "→", d.page_content[:200], "\n")

## 4. Agente único (v5)
Toda cifra sale de DuckDB; el LLM solo decide **qué** consultar e interpreta. Antes de ejecutar cada herramienta,
el middleware `limpiar_argumentos` (`src/guardarrailes.py`) corrige y valida los argumentos.

| Herramienta | Uso |
|---|---|
| `consultar_metricas` | métricas y KPIs con filtros y agrupación |
| `ranking` | top/bottom N por KPI |
| `detectar_anomalias` | problemas de entrega y tracking |
| `buscar_conocimiento` | RAG: definiciones, brief, benchmarks |
| `generar_grafico` | gráfico Plotly (línea si el eje es temporal, barras si es categórico) |

In [ ]:
from src.agent import crear_agente, preguntar, figuras_del_turno

agente = crear_agente(LLM_PRINCIPAL)

def mostrar(pregunta, thread="demo"):
    """Agente único: imprime el razonamiento (herramientas) y muestra los gráficos."""
    out = agente.invoke({"messages": [{"role": "user", "content": pregunta}]},
                        {"configurable": {"thread_id": thread}})
    msgs = out["messages"]
    inicio = max(i for i, m in enumerate(msgs) if m.type == "human")
    for m in msgs[inicio:]:
        m.pretty_print()
    for fig_json in figuras_del_turno(msgs):
        pio.from_json(fig_json).show()

mostrar("¿Qué plataforma tuvo el menor costo por cita en US durante agosto?", thread="t1")

In [ ]:
mostrar("¿Detectas problemas de tracking o gasto sin entrega en la campaña?", thread="t2")

In [ ]:
mostrar("¿Qué es el VTR y cuál fue el de Snapchat comparado con su benchmark interno?", thread="t3")

In [ ]:
mostrar("Grafica la evolución semanal del costo por cita por plataforma en US", thread="t4")

👉 En LangSmith se ve el árbol de cada respuesta: modelo → herramienta(s) → modelo.

## 5. Memoria conversacional
Mismo `thread_id` → el agente recuerda el contexto (preguntas de seguimiento).

In [ ]:
print(preguntar(agente, "¿Cuál fue el CTR de TikTok en julio?", thread_id="memoria-1"))
print("---")
print(preguntar(agente, "¿Y en agosto? ¿Subió o bajó?", thread_id="memoria-1"))

## 6. Arquitectura multiagente
El agente único llegó a 13/14 respuestas numéricas correctas, pero falló en la pregunta **mixta** (comparar una cifra
contra el objetivo del brief): necesitaba encadenar dos herramientas y comparar. El multiagente reparte el trabajo entre
especialistas y agrega una revisión antes de entregar.

**Patrones de diseño usados**

| Patrón | Dónde | Para qué |
|---|---|---|
| **Router** | Supervisor | Clasifica la pregunta (datos, anomalías, conocimiento, mixta) y la envía al especialista adecuado |
| **Paralelización** | Preguntas mixtas | Datos y Conocimiento trabajan a la vez; el redactor une la cifra y el objetivo |
| **Evaluador–optimizador** | Crítico + redactor | El crítico revisa; si rechaza, el redactor reescribe con su retroalimentación |

**Decisiones de diseño**

| Decisión | Elección |
|---|---|
| Control del flujo | Grafo fijo en LangGraph; el LLM solo decide la ruta y si el crítico aprueba |
| Comunicación | Mensajes estructurados con Pydantic (`Enrutamiento`, `Veredicto`) |
| Memoria | Estado compartido del grafo + checkpointer por conversación |
| Modelo por rol | Qwen 2.5 7B en todos los roles, para que la diferencia venga del diseño y no del modelo |
| Herramientas | Cada especialista ve solo las suyas; todos comparten el middleware de guardarraíles de v5 |
| Criterio de parada | Crítico: máximo 3 rondas o tope de costo por pregunta |
| Crítico | Verificación determinista de cifras (0 tokens) siempre + LLM solo en preguntas mixtas |

In [ ]:
from src.grafo import crear_multiagente, responder
from src.costos import ContadorCostos

grafo = crear_multiagente(LLM_PRINCIPAL)

# Diagrama del grafo (la imagen usa mermaid.ink; si no hay internet se imprime el texto Mermaid)
from IPython.display import Image, display
try:
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception:
    print(grafo.get_graph().draw_mermaid())

In [ ]:
def mostrar_multi(pregunta, thread="multi-demo"):
    """Multiagente: ruta, herramientas, veredicto del crítico, respuesta, gráficos y costo del turno."""
    contador = ContadorCostos(agente_por_defecto="otro")
    r = responder(grafo, pregunta, thread_id=thread, callbacks=[contador])
    c = contador.resumen()
    print(f"Pregunta: {pregunta}")
    print(f"  Ruta: {r['ruta']}  ({r['razon_ruta']})")
    print(f"  Agentes: {' → '.join(c['agentes'])} | herramientas: {r['herramientas']}")
    print(f"  Crítico: {r['veredicto']} en {r['rondas']} ronda(s)")
    print(f"  Costo: {c['llamadas']} llamadas · {c['tokens_entrada'] + c['tokens_salida']:,} tokens · "
          f"{c['segundos_llm']:.1f} s · USD {c['usd_sombra']:.5f} sombra\n")
    print(r["respuesta"], "\n" + "-" * 80)
    for f in r["figuras"]:
        pio.from_json(f).show()
    return r

for i, p in enumerate(["¿Cuál fue el CPM en CA?",
                       "En agosto, en US, ¿qué plataforma tuvo el menor costo por cita?",
                       "¿El lower funnel cumplió el objetivo de costo por cita?",
                       "¿Detectas gasto sin entrega en septiembre?"]):
    mostrar_multi(p, thread=f"multi-{i}")

In [ ]:
# La misma pregunta de gráfico que en la sección 4, ahora con el multiagente
mostrar_multi("Grafica la evolución semanal del costo por cita por plataforma en US", thread="multi-grafico")

In [ ]:
# Memoria en el multiagente: el supervisor y los especialistas ven la conversación previa
mostrar_multi("¿Cuál fue el CTR de TikTok en julio?", thread="multi-memoria")
mostrar_multi("¿Y en agosto? ¿Subió o bajó?", thread="multi-memoria");

👉 En LangSmith, cada respuesta del multiagente muestra el árbol supervisor → especialista(s) → crítico, con los tokens de cada agente.

## 7. Cálculo de costo
Fórmula por llamada (precios en USD por millón de tokens):

$$\text{costo}_{llamada} = \text{tokens}_{entrada} \times p_{entrada} + \text{tokens}_{salida} \times p_{salida}$$

El costo de una pregunta es la suma de todas las llamadas de todos los agentes, incluidas las rondas del crítico.
Como Qwen corre local, se reportan tres costos (`src/costos.py`):

| Costo | Qué mide |
|---|---|
| **USD real** | Lo que cobra un proveedor de pago (aquí 0: todo es local) |
| **USD sombra** | Lo que costarían los mismos tokens con un modelo de nube de referencia (precios de clase, oct. 2026) |
| **Energía** | kWh y COP de la GPU mientras Ollama genera: potencia × tiempo × tarifa |

El callback `ContadorCostos` lee los tokens que reporta Ollama en cada llamada y los atribuye al agente que la hizo.

In [ ]:
from src.config import PRECIOS_USD_POR_MILLON, POTENCIA_GPU_W, TARIFA_KWH_COP
pd.DataFrame(PRECIOS_USD_POR_MILLON, index=["entrada USD/M", "salida USD/M"]).T.assign(
    referencia_sombra=lambda d: d.index == MODELO_REFERENCIA_SOMBRA)

In [ ]:
# Mismas 3 preguntas en las dos arquitecturas
PREGUNTAS_COSTO = ["¿Cuál fue el CPM en CA?",
                   "¿Qué plataforma tuvo el menor costo por cita en agosto?",
                   "¿El lower funnel cumplió el objetivo de costo por cita?"]

c_unico = ContadorCostos(agente_por_defecto="agente_unico")
for i, p in enumerate(PREGUNTAS_COSTO):
    agente.invoke({"messages": [{"role": "user", "content": p}]},
                  {"configurable": {"thread_id": f"costo-unico-{i}"}, "callbacks": [c_unico]})

c_multi = ContadorCostos(agente_por_defecto="otro")
for i, p in enumerate(PREGUNTAS_COSTO):
    responder(grafo, p, thread_id=f"costo-multi-{i}", callbacks=[c_multi])

campos = ["llamadas", "tokens_entrada", "tokens_salida", "segundos_llm", "usd_sombra", "kwh", "cop_energia"]
comparacion_costo = pd.DataFrame({"Agente único": c_unico.resumen(), "Multiagente": c_multi.resumen()}).loc[campos]
comparacion_costo["Multi / único"] = (comparacion_costo["Multiagente"].astype(float)
                                      / comparacion_costo["Agente único"].astype(float)).round(2)
comparacion_costo

In [ ]:
# Costo por agente del multiagente (tabla de la diapositiva 8) con los 4 modelos de referencia
c_multi.tabla()

In [ ]:
# Overhead de coordinación: tokens que gastan supervisor, redactor y crítico frente al trabajo de los especialistas
d = c_multi.df()
d["tokens"] = d.entrada + d.salida
coordinacion = d[d.agente.isin(["supervisor", "redactor", "critico"])].tokens.sum()
print(f"Tokens de coordinación: {coordinacion:,} de {d.tokens.sum():,} ({coordinacion / d.tokens.sum():.0%})")
print(f"Entrada media por llamada de especialista: {d[d.agente.isin(['datos','anomalias','conocimiento'])].entrada.mean():,.0f} tokens"
      f" vs agente único: {c_unico.df().entrada.mean():,.0f}")

## 8. Evaluación comparativa en LangSmith
Dataset `ben-frank-asistente-eval`: 17 preguntas con respuestas de referencia **calculadas desde DuckDB** (verdad exacta).

| Evaluador | Qué mide | Tipo |
|---|---|---|
| `valor_correcto` | La cifra o entidad de referencia aparece en la respuesta (±1 %) — **métrica principal** | Determinista |
| `herramienta_correcta_v2` | Usó todas las herramientas requeridas (las dos en la mixta) | Determinista |
| `juez_externo` | Un LLM de **otra familia** (Llama 3.1 8B local) juzga contra la referencia | LLM |
| `costo_usd_sombra`, `tokens_totales`, `llamadas_llm` | Costo de cada respuesta | Medición |

Cada pregunta se corre varias veces (`REPETICIONES`) porque, aun con `temperature=0`, el resultado cambia entre corridas.

In [ ]:
from src.construir_eval import construir
from src.evaluate import subir_dataset

construir()        # recalcula las respuestas de referencia desde DuckDB
subir_dataset()    # no duplica si el dataset ya existe

In [ ]:
# Prueba rápida del juez antes de la corrida larga
from src.evaluacion_multi import juez_externo, juez_activo
print("Juez:", juez_activo())
juez_externo({"pregunta": "¿CPM en CA?"}, {"respuesta": "Fue $5.66"}, {"respuesta": "$5.66"})

In [ ]:
# Corrida completa: ~30 min con 4 configuraciones × 17 preguntas × 2 repeticiones.
# Para una prueba corta: deja solo las 2 primeras configuraciones y REPETICIONES = 1.
from src.evaluacion_multi import (correr, resumen_comparativo, variabilidad, costo_juez,
                                  target_agente_unico, target_multiagente)

REPETICIONES = 2
CONFIGURACIONES = {
    "v5 agente único":         ("v5-agente-unico",   lambda: target_agente_unico()),
    "multiagente":             ("multiagente-v1",    lambda: target_multiagente()),
    "multi · router reglas":   ("multi-reglas",      lambda: target_multiagente(router_modo="reglas")),
    "multi · sin crítico LLM": ("multi-sin-critico", lambda: target_multiagente(critico_llm="nunca")),
}

resultados = {}
for nombre, (prefijo, crear_target) in CONFIGURACIONES.items():
    print(f"▶ {nombre}")
    resultados[nombre] = correr(prefijo, crear_target(), repeticiones=REPETICIONES)

# Guardar para el informe (y para no repetir la corrida)
SALIDA = ROOT / "eval" / "resultados"
SALIDA.mkdir(parents=True, exist_ok=True)
for nombre, res in resultados.items():
    res.to_pandas().to_csv(SALIDA / f"{nombre.replace(' ', '_').replace('·', '')}.csv", index=False)

In [ ]:
# Tabla comparativa: calidad, robustez, latencia y costo por configuración
resumen = resumen_comparativo(resultados)
resumen.to_csv(SALIDA / "resumen_comparativo.csv")
resumen

In [ ]:
# Calidad por tipo de pregunta (dónde gana o pierde cada arquitectura)
def por_tipo(res):
    df = res.to_pandas()
    df["ok"] = df["feedback.valor_correcto"].map({True: 1.0, False: 0.0})
    return df.dropna(subset=["ok"]).groupby("reference.tipo")["ok"].mean()

pd.DataFrame({n: por_tipo(r) for n, r in resultados.items()}).style.format("{:.0%}")

In [ ]:
# Preguntas cuyo resultado cambió entre repeticiones (variabilidad)
for nombre, res in resultados.items():
    v = variabilidad(res)
    print(f"{nombre}: {len(v)} pregunta(s) inestables")
    if len(v):
        display(v)

In [ ]:
# Costo del juez durante la evaluación (sombra, porque corre local)
costo_juez()

## 9. Extra: informe ejecutivo con salida estructurada
Muestra `with_structured_output`: el LLM devuelve un objeto Pydantic con campos fijos en vez de texto libre.

In [ ]:
from pydantic import BaseModel, Field
from langsmith import traceable

class InformeCampana(BaseModel):
    titulo: str
    resumen: str = Field(description="2-3 frases")
    hallazgos: list[str] = Field(description="3 a 5 hallazgos con cifras")
    recomendaciones: list[str] = Field(description="2 a 3 acciones concretas")

@traceable(name="informe_ejecutivo", tags=["informe"])
def informe_ejecutivo(site=None, pais=None):
    datos = T.a_texto(T.consultar_metricas(["funnel"], site=site, pais=pais))
    anom = T.a_texto(T.detectar_anomalias(site=site, pais=pais))
    redactor = model.with_structured_output(InformeCampana)
    return redactor.invoke(
        f"Eres analista de medios de Ben & Frank. Con estos datos (no inventes cifras) escribe un informe.\n"
        f"Filtro: site={site}, pais={pais}\nKPIs por funnel:\n{datos}\nAnomalías:\n{anom}")

# informe_ejecutivo(site="facebook", pais="US")

## 10. Anexo: depuración de argumentos
`REGISTRO` guarda, para cada llamada a herramienta, lo que **envió el modelo** y lo que **se ejecutó** después de los
guardarraíles. Así se diagnosticaron los errores de v1–v5 y el de anomalías del multiagente (listas en `site` y `pais`).

In [ ]:
from src.guardarrailes import REGISTRO

def diagnosticar(preguntas, sistema="multi"):
    for i, q in enumerate(preguntas):
        REGISTRO.clear()
        if sistema == "multi":
            resp = responder(grafo, q, thread_id=f"diag-{sistema}-{i}")["respuesta"]
        else:
            resp = preguntar(agente, q, thread_id=f"diag-{sistema}-{i}")
        print("P:", q)
        for r in REGISTRO:
            print(f"   [{r['herramienta']}] modelo envió: {r['modelo_envio']}")
            print(f"   {' ' * (len(r['herramienta']) + 2)} se ejecutó:   {r['ejecutado']}")
        print("   R:", resp[:250], "\n" + "-" * 70)

diagnosticar(["¿Detectas gasto sin entrega en septiembre?",
              "¿Qué mes generó más citas de examen visual?"])

In [ ]:
# Respuestas correctas calculadas directo con la capa determinista (sin LLM)
print(T.ranking("site", kpi="costo_por_cita", n=3, pais="US", fecha_inicio="2026-08-01", fecha_fin="2026-08-31"))
print(T.ranking("creative", kpi="vtr", n=3, mejores=False, site="tiktok"))
print(T.detectar_anomalias(fecha_inicio="2026-09-01", fecha_fin="2026-09-30"))

In [ ]:
# Prueba directa de la herramienta de gráficos (sin LLM)
from src.graficos import grafico
fig, datos = grafico("ctr", por="site", funnel="middle")
fig.show()

## Próximos pasos
- Interfaz: `streamlit run app.py` (selector agente único / multiagente, ruta de agentes y costo por respuesta).
- Resultados de la sección 8 → informe del proyecto.